# Integrate literature → `kg_rd` (`integrate_primekg_plus_rd.py`)

This notebook runs the integration pipeline and **explains why many rows are still skipped** after two rounds of expert review.

## Input pool

| Source | File | Notes |
|-------|------|---------|
| Algorithm final (v1) | `THUY_DATA_CURATION/20260508-*_final.csv` | entity_status: `in_kg`, CUI, `invalid` |
| Expert delta (v2) | `Post curation/merged_expert_v2/*_additional_relations_v2.csv` | only `integratable=True`; status includes assigned CUI |

## Output

- `primekg_plus_rd.csv` = `primekg_plus.csv` + novel literature edges
- `primekg_plus_rd_nodes.csv` = `nodes.csv` + literature nodes (MONDO/HPO from CUI)
- Audit: integrated / skipped / summary JSON

## Understanding skip counts

**Skip ≠ failed expert review.** A row is skipped because:

1. **`duplicate_literature_row`** — duplicate edge in the pool (final + v2 overlap, or same PMID/entity/relation)
2. **`unsupported_relation:*`** — relation type **not in the PrimeKG schema** (e.g. `anatomy_disease`, `drug_bioprocess`)
3. **`unresolved_entity*`** — endpoint cannot be mapped (gene/drug/GO lacks a CUI bridge, or CUI has no MONDO/HPO)
4. **`already_in_base_kg`** — integrated but **not counted as novel** (edge already in `primekg_plus.csv`)

Run the cells below for a detailed breakdown.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

SCRIPT_DIR = Path.cwd()
if not (SCRIPT_DIR / "integrate_primekg_plus_rd.py").exists():
    SCRIPT_DIR = Path("/Users/ljw303/YANG_DATA/PrimeKG/PrimeKG-Plus_release/scripts/literature_curation")
sys.path.insert(0, str(SCRIPT_DIR))

from integrate_primekg_plus_rd import (
    collect_literature_rows,
    default_disease_specs,
    integrate_literature,
    build_node_index_lookup,
    index_literature_edges,
    resolve_primekg_root,
    resolve_release_root,
)
from lib.entity_resolver import EntityResolver, parse_cuis, ENTITY_TYPE_TO_NODE
from lib.relation_config import KG_OUTPUT_COLUMNS

RUN_DATE = os.environ.get("RUN_DATE", "20260529")
DRY_RUN = True  # set False to write CSV outputs

primekg_root = resolve_primekg_root()
release_root = resolve_release_root()
curation_root = Path(os.environ.get("CURATION_ROOT", "/Users/ljw303/YANG_DATA/THUY_DATA_CURATION"))
data_dir = primekg_root / "datasets" / "data"
plus_kg = Path(os.environ.get("PLUS_KG", release_root / "dataset" / "kg.csv"))
plus_nodes = Path(os.environ.get("PLUS_NODES", release_root / "dataset" / "nodes.csv"))
out_dir = Path(os.environ.get("KG_RD_OUT_DIR", release_root / "dataset" / "literature_curation"))

print("PrimeKG root:", primekg_root)
print("Release root:", release_root)
print("Curation root:", curation_root)
print("Output dir:", out_dir)


In [ ]:
base_kg = pd.read_csv(plus_kg, low_memory=False)
nodes = pd.read_csv(plus_nodes, low_memory=False)
specs = default_disease_specs(curation_root)
literature = collect_literature_rows(specs)

print(f"Base kg:   {len(base_kg):,} edges, {len(nodes):,} nodes")
print(f"Pool:      {len(literature):,} curated rows (deduped)")
print("\nRows by source file:")
display(literature.groupby(["disease_cohort", "source_file"]).size().reset_index(name="n").sort_values("n", ascending=False))


In [ ]:
resolver = EntityResolver(nodes, data_dir)
lit_edges, provenance, skipped = integrate_literature(literature, resolver, base_kg)

novel_mask = ~provenance["already_in_base_kg"] if len(provenance) else pd.Series(dtype=bool)
novel_lit = lit_edges.loc[novel_mask.values].reset_index(drop=True) if len(lit_edges) else lit_edges
nodes_rd = resolver.extend_nodes(nodes)
lookup = build_node_index_lookup(nodes_rd)
novel_indexed = index_literature_edges(novel_lit, lookup) if len(novel_lit) else pd.DataFrame(columns=KG_OUTPUT_COLUMNS)

summary = {
    "curated_input_rows": len(literature),
    "integrated_rows": len(lit_edges),
    "novel_edges": len(novel_lit),
    "skipped_rows": len(skipped),
    "literature_new_nodes": len(resolver.new_nodes),
    "kg_rd_nodes": len(nodes_rd),
    "kg_rd_edges": len(base_kg) + len(novel_indexed),
}
pd.Series(summary).to_frame("count")


## 1. Skip reasons — by category

In [ ]:
skip = skipped.copy()
skip["category"] = skip["reason"].map(lambda r: {
    "duplicate_literature_row": "A_duplicate_in_pool",
}.get(r, "B_unresolved" if r.startswith("unresolved") else (
    "C_unsupported_schema" if r.startswith("unsupported") else (
    "D_unknown_relation" if r.startswith("unknown") else "E_other"
))))

cat = skip.groupby("category").size().sort_values(ascending=False)
cat_df = cat.to_frame("rows").assign(pct=lambda x: (100 * x.rows / len(skip)).round(1))
display(cat_df)

print("\nDetail by reason:")
display(skip["reason"].value_counts().reset_index(name="rows"))


**Quick interpretation:**

- **A_duplicate** — edge already present; not a mapping error
- **C_unsupported_schema** — experts accepted the relation but PrimeKG has **no matching edge type**
- **B_unresolved** — endpoint truly unmapped → see section 2


## 2. Unresolved endpoints — detail

In [ ]:
def unresolved_detail(sk: pd.DataFrame, lit: pd.DataFrame, resolver: EntityResolver) -> pd.DataFrame:
    rows = []
    unres = sk[sk["reason"].str.startswith("unresolved")]
    for _, s in unres.iterrows():
        row = lit.loc[int(s["row_index"])]
        side = 1 if s["reason"] == "unresolved_entity1" else 2
        cols = {
            1: ("entity1", "entity1_status", "entity_type1", "entity1_suggested_name", "entity1_expert_cui"),
            2: ("entity2", "entity2_status", "entity_type2", "entity2_suggested_name", "entity2_expert_cui"),
        }
        name, status, etype, sug, expert = [row.get(c) for c in cols[side]]
        cuis = parse_cuis(expert) or parse_cuis(status)
        etype = str(etype or "")
        bridge = etype in ("disease", "phenotype")
        oid = None
        if cuis and bridge:
            m = resolver._cui_to_mondo if etype == "disease" else resolver._cui_to_hp
            for c in cuis:
                oid = m.get(c)
                if oid:
                    break
        rows.append({
            "disease": s["disease_cohort"],
            "source_file": s["source_file"],
            "entity": name,
            "entity_type": etype,
            "status": status,
            "expert_cui": expert,
            "has_cui": bool(cuis),
            "cui_bridge_type": bridge,
            "ontology_id": oid,
            "PMID": s.get("PMID"),
            "Relation": s.get("Relation"),
        })
    return pd.DataFrame(rows)

unres_df = unresolved_detail(skipped, literature, resolver)
print("Unresolved endpoint events:", len(unres_df))
print("\nBy entity_type:")
display(unres_df["entity_type"].value_counts().reset_index(name="n"))
print("\nCUI present but entity_type without MONDO/HPO bridge (gene/drug/GO/...):")
mask = unres_df["has_cui"] & ~unres_df["cui_bridge_type"]
print(mask.sum(), "events")
display(unres_df[mask].groupby("entity_type").size().reset_index(name="n").sort_values("n", ascending=False))


In [ ]:
print("Sample unresolved — has CUI but type has no MONDO/HPO bridge:")
display(unres_df[unres_df["has_cui"] & ~unres_df["cui_bridge_type"]].head(20))

print("\nSample unresolved — disease/phenotype with CUI + ontology id:")
display(unres_df[unres_df["cui_bridge_type"] & unres_df["ontology_id"].notna()].head(20))

print("\nSample unresolved — no CUI:")
display(unres_df[~unres_df["has_cui"]].head(15))


## 3. Integrated rows — method breakdown

In [ ]:
if len(provenance):
    m1 = provenance["entity1_resolve_method"].value_counts()
    m2 = provenance["entity2_resolve_method"].value_counts()
    display(pd.DataFrame({"entity1": m1, "entity2": m2}).fillna(0).astype(int))

print("Novel vs already in base kg:")
display(provenance["already_in_base_kg"].value_counts().rename({True: "already_in_kg", False: "novel_literature"}))

if resolver.new_nodes:
    print("\nLiterature nodes added:")
    display(pd.DataFrame(resolver.new_nodes))


## 4. Skip vs source file — final v1 or expert v2?

In [ ]:
skip_src = skipped.groupby(["source_file", "reason"]).size().reset_index(name="n")
skip_src = skip_src.sort_values(["source_file", "n"], ascending=[True, False])
display(skip_src.head(40))

# integratable flag only on v2 files
v2_only = literature[literature["source_file"].str.contains("additional", na=False)]
print(f"\nV2 additional rows in pool: {len(v2_only)}")
if "integratable" in v2_only.columns:
    print(v2_only["integratable"].value_counts())


## 5. Unsupported relations — schema gap (not an expert error)

In [ ]:
from lib.relation_config import UNSUPPORTED_RELATIONS
unsupported = skipped[skipped["reason"].str.startswith("unsupported")]
print("PrimeKG UNSUPPORTED_RELATIONS list:")
print(sorted(UNSUPPORTED_RELATIONS))
print("\nSkipped unsupported in this pool:")
display(unsupported.groupby(skipped["reason"]).size().reset_index(name="n"))
display(unsupported[["disease_cohort", "entity1", "entity2", "Relation", "PMID"]].head(20))


## 6. (Optional) Write outputs — same as `integrate_primekg_plus_rd.py`

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True — not writing files. Set DRY_RUN=False to write.")
else:
    import shutil
    out_dir.mkdir(parents=True, exist_ok=True)
    tag = RUN_DATE
    kg_rd = pd.concat([base_kg[KG_OUTPUT_COLUMNS], novel_indexed], ignore_index=True)
    kg_rd.to_csv(out_dir / f"{tag}-primekg_plus_rd.csv", index=False)
    nodes_rd.to_csv(out_dir / f"{tag}-primekg_plus_rd_nodes.csv", index=False)
    provenance.to_csv(out_dir / f"{tag}-literature_edges_integrated.csv", index=False)
    skipped.to_csv(out_dir / f"{tag}-literature_edges_skipped.csv", index=False)
    if resolver.new_nodes:
        pd.DataFrame(resolver.new_nodes).to_csv(out_dir / f"{tag}-literature_nodes_added.csv", index=False)
    print("Wrote outputs to", out_dir)
